# Spin-Echo Simulation — Analysis

**BlochSimulator Version**: 1.0.15

**Mode**: Load data from HDF5 file

**Data file**: `spin_echo_analysis_data.h5`

This notebook loads pre-computed simulation data and provides visualization and analysis tools.

## Installation

If you haven't installed the `blochsimulator` package yet, you can do so using pip:

```bash
# From GitHub (latest version)
!pip install git+https://github.com/LucaNagel/bloch_sim_gui.git

# From local directory (if you have the source code)
# !pip install .
```

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import xarray as xr
from pathlib import Path
from blochsimulator import BlochSimulator

# Set matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Load Simulation Data

In [ ]:
# Load data from HDF5 file
data_file = 'spin_echo_analysis_data.h5'

if not Path(data_file).exists():
    raise FileNotFoundError(f"Data file not found: {data_file}")

print(f"Loading data from: {data_file}")

# Initialize simulator to handle data loading
sim = BlochSimulator()
sim.load_results(data_file)
data = sim.last_result

# Convert tissue to dictionary for consistent access
from dataclasses import asdict
if hasattr(data['tissue'], '__dataclass_fields__'):
    data['tissue'] = asdict(data['tissue'])

# Load additional parameters (metadata) not loaded by the simulator core
with h5py.File(data_file, 'r') as f:
    # Load sequence parameters
    data['sequence_params'] = {}
    if 'sequence_parameters' in f:
        grp = f['sequence_parameters']
        for key in grp.attrs.keys():
            data['sequence_params'][key] = grp.attrs[key]
        for key in grp.keys():
            if isinstance(grp[key], h5py.Dataset):
                data['sequence_params'][key] = grp[key][...]

    # Load simulation parameters
    data['simulation_params'] = {}
    if 'simulation_parameters' in f:
        grp = f['simulation_parameters']
        for key in grp.attrs.keys():
            data['simulation_params'][key] = grp.attrs[key]
        for key in grp.keys():
            if isinstance(grp[key], h5py.Dataset):
                data['simulation_params'][key] = grp[key][...]

print(f"Data loaded successfully!")
if 'mx' in data:
    print(f"  Shape: {data['mx'].shape}")
if 'time' in data:
    print(f"  Duration: {data['time'][-1]*1000:.3f} ms")


## Xarray Dataset

In [ ]:
# Convert to xarray Dataset for advanced analysis
# Extract info from metadata
n_pos = data.get('simulation_params', {}).get('num_positions', 1)
n_freq = data.get('simulation_params', {}).get('num_frequencies', 1)
time = data.get('time')
n_time = len(time) if time is not None else 0

# Create DataArray for each component
vars = {}
coords = {}
if time is not None: coords['time'] = time

for k in ['mx', 'my', 'mz', 'signal']:
    v = data[k]
    dims = []

    # Try to intelligently name dimensions
    for i, dim_len in enumerate(v.shape):
        if n_time > 0 and dim_len == n_time:
            dims.append('time')
        elif n_pos > 1 and dim_len == n_pos:
            dims.append('position')
        elif n_freq > 1 and dim_len == n_freq:
            dims.append('frequency')
        else:
            dims.append(f'dim_{i}')

    vars[k] = (dims, v)

ds = xr.Dataset(vars, coords=coords)
# Add metadata
ds.attrs.update(data.get('simulation_params', {}))
ds.attrs.update(data.get('sequence_params', {}))

print('Xarray Dataset created as "ds":')
print(ds)

## Simulation Parameters

In [ ]:
# Display simulation parameters
print("="*60)
print("SIMULATION PARAMETERS")
print("="*60)

print("\nTissue:")
for key, value in data['tissue'].items():
    if key in ['t1', 't2', 't2_star'] and value is not None:
        print(f"  {key}: {value*1000:.1f} ms")
    elif value is not None:
        print(f"  {key}: {value}")

print("\nSequence:")
for key, value in data['sequence_params'].items():
    if not isinstance(value, np.ndarray):
        print(f"  {key}: {value}")

print("\nSimulation:")
for key, value in data['simulation_params'].items():
    if not isinstance(value, np.ndarray):
        print(f"  {key}: {value}")

print("="*60)


## Quick Analysis

In [ ]:
# Quick analysis
print("\nData Statistics:")
print(f"  Time points: {len(data['time'])}")
print(f"  Positions: {data['positions'].shape[0]}")
print(f"  Frequencies: {len(data['frequencies'])}")

if data['mx'].ndim == 3:  # Time-resolved
    mx_final = data['mx'][-1]
    my_final = data['my'][-1]
    mz_final = data['mz'][-1]

    print("\nFinal Magnetization:")
    print(f"  Mx range: [{mx_final.min():.4f}, {mx_final.max():.4f}]")
    print(f"  My range: [{my_final.min():.4f}, {my_final.max():.4f}]")
    print(f"  Mz range: [{mz_final.min():.4f}, {mz_final.max():.4f}]")

    # Find peak transverse magnetization
    mxy = np.sqrt(data['mx']**2 + data['my']**2)
    max_mxy = mxy.max()
    max_idx = np.unravel_index(mxy.argmax(), mxy.shape)

    print(f"\n  Peak |Mxy|: {max_mxy:.4f}")
    print(f"  At time: {data['time'][max_idx[0]]*1000:.3f} ms")


## Magnetization Evolution

In [ ]:
# Plot magnetization evolution
# Always choose central index for position and frequency
position_idx = data['positions'].shape[0] // 2
freq_idx = len(data['frequencies']) // 2

# Get actual values for title
pos_z_cm = data['positions'][position_idx, 2] * 100
freq_hz = data['frequencies'][freq_idx]

if data['mx'].ndim == 3:  # Time-resolved
    time_ms = data['time'] * 1000
    mx = data['mx'][:, position_idx, freq_idx]
    my = data['my'][:, position_idx, freq_idx]
    mz = data['mz'][:, position_idx, freq_idx]
    mxy = np.sqrt(mx**2 + my**2)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0, 0].plot(time_ms, mx, 'b-', linewidth=1.5)
    axes[0, 0].set_xlabel('Time (ms)')
    axes[0, 0].set_ylabel('Mx')
    axes[0, 0].set_title('Transverse Magnetization (x)')
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(time_ms, my, 'r-', linewidth=1.5)
    axes[0, 1].set_xlabel('Time (ms)')
    axes[0, 1].set_ylabel('My')
    axes[0, 1].set_title('Transverse Magnetization (y)')
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(time_ms, mz, 'g-', linewidth=1.5)
    axes[1, 0].set_xlabel('Time (ms)')
    axes[1, 0].set_ylabel('Mz')
    axes[1, 0].set_title('Longitudinal Magnetization')
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(time_ms, mxy, color='purple', linewidth=1.5)
    axes[1, 1].set_xlabel('Time (ms)')
    axes[1, 1].set_ylabel('|Mxy|')
    axes[1, 1].set_title('Transverse Magnitude')
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(f'Magnetization Evolution - Pos: {pos_z_cm:.2f} cm, Freq: {freq_hz:.1f} Hz',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Endpoint data - no time evolution to plot")


## MRI Signal

In [ ]:
# Plot signal
# Re-use central indices
position_idx = data['positions'].shape[0] // 2
freq_idx = len(data['frequencies']) // 2

pos_z_cm = data['positions'][position_idx, 2] * 100
freq_hz = data['frequencies'][freq_idx]

if data['signal'].ndim == 3:  # Time-resolved
    signal = data['signal'][:, position_idx, freq_idx]
    time_ms = data['time'] * 1000

    fig, axes = plt.subplots(2, 1, figsize=(12, 8))

    axes[0].plot(time_ms, np.real(signal), 'b-', label='Real', linewidth=1.5)
    axes[0].plot(time_ms, np.imag(signal), 'r-', label='Imaginary', linewidth=1.5)
    axes[0].set_xlabel('Time (ms)')
    axes[0].set_ylabel('Signal')
    axes[0].set_title('Complex Signal Components')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(time_ms, np.abs(signal), color='purple', linewidth=1.5)
    axes[1].set_xlabel('Time (ms)')
    axes[1].set_ylabel('|Signal|')
    axes[1].set_title('Signal Magnitude')
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(f'MRI Signal - Pos: {pos_z_cm:.2f} cm, Freq: {freq_hz:.1f} Hz',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Endpoint data - no time evolution to plot")


## Spatial Profile

In [ ]:
# Plot spatial profile
time_idx = -1  # Final time point
freq_idx = 0

if data['mz'].ndim == 3:
    mz = data['mz'][time_idx, :, freq_idx]
    mx = data['mx'][time_idx, :, freq_idx]
    my = data['my'][time_idx, :, freq_idx]
elif data['mz'].ndim == 2:
    mz = data['mz'][:, freq_idx]
    mx = data['mx'][:, freq_idx]
    my = data['my'][:, freq_idx]

mxy = np.sqrt(mx**2 + my**2)
z_pos = data['positions'][:, 2] * 100  # Convert to cm

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(z_pos, mz, 'go-', linewidth=2, markersize=6)
ax1.set_xlabel('Position (cm)')
ax1.set_ylabel('Mz')
ax1.set_title('Longitudinal Magnetization Profile')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.3)

ax2.plot(z_pos, mxy, 'mo-', linewidth=2, markersize=6)
ax2.set_xlabel('Position (cm)')
ax2.set_ylabel('|Mxy|')
ax2.set_title('Transverse Magnetization Profile')
ax2.grid(True, alpha=0.3)

freq = data['frequencies'][freq_idx]
plt.suptitle(f'Spatial Profile - Frequency: {freq:.1f} Hz',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## Custom Analysis

Add your custom analysis code here. Available data:
- `data['mx']`, `data['my']`, `data['mz']` - Magnetization components
- `data['signal']` - Complex signal
- `data['time']` - Time points
- `data['positions']` - Spatial positions
- `data['frequencies']` - Off-resonance frequencies

In [ ]:
# Your custom analysis code here
